# Analise exploratoria dos dados telefonicos de Lavras

Este notebook documenta e expande a exploracao inicial do projeto. A base principal, `Lavras.parquet`, parece estar agregada por emissor residente em Lavras: cada linha contem um usuario emissor, totais gerais e listas com informacoes dos receptores correspondentes.

A base `residencias.csv` e muito maior e contem informacoes de residencia e quintis. Por isso, a exploracao dela e feita por amostra ou leitura em partes, evitando carregar quase 1 GB de dados de uma vez.

## Perguntas iniciais

- Quantos emissores existem na amostra de Lavras?
- Como se distribuem o numero de chamadas, receptores unicos, distancias e duracoes?
- Ha usuarios muito mais ativos que o restante?
- As listas internas do parquet sao consistentes com a coluna `unique_receivers`?
- Que estrutura de rede surge ao transformar emissores e receptores em arestas?
- Como a base grande de residencias se distribui por cidade e quintis em uma amostra?

In [1]:
from pathlib import Path
import sys

# Evita importar extensoes opcionais incompatíveis com NumPy 2.x neste ambiente.
sys.modules['numexpr'] = None
sys.modules['bottleneck'] = None

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.graph_objects as go
import networkx as nx

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

DATA_DIR = Path('.')
LAVRAS_PATH = DATA_DIR / 'Lavras.parquet'
RESIDENCIAS_PATH = DATA_DIR / 'residencias.csv'

LAVRAS_PATH.exists(), RESIDENCIAS_PATH.exists()

ModuleNotFoundError: No module named 'pyarrow'

## Dicionario inicial de dados

| Coluna | Descricao inferida |
|---|---|
| `city_residences` | Cidade de residencia do emissor. Neste recorte, deve ser Lavras. |
| `id_emisor` | Identificador anonimizado do usuario emissor. |
| `total_calls_make` | Total de chamadas feitas pelo emissor. |
| `unique_receivers` | Quantidade de receptores distintos chamados pelo emissor. |
| `IDs_receivers_corr` | Lista de identificadores anonimizados dos receptores. |
| `q_calls_corr` | Lista com quantidade de chamadas para cada receptor correspondente. |
| `residence_distance_km_corr` | Lista com distancia residencial, em km, entre emissor e receptor correspondente. |
| `calls_duration_total_corr` | Lista com duracao total das chamadas para cada receptor correspondente. |

As colunas com sufixo `_corr` precisam ter o mesmo comprimento linha a linha, pois representam atributos de uma mesma aresta emissor-receptor.

In [ ]:
parquet_file = pq.ParquetFile(LAVRAS_PATH)
print(parquet_file.schema)
print(f'Linhas: {parquet_file.metadata.num_rows:,}')
print(f'Grupos de linhas: {parquet_file.metadata.num_row_groups}')

required group field_id=-1 schema {
  optional binary field_id=-1 city_residences (String);
  optional binary field_id=-1 id_emisor (String);
  optional int64 field_id=-1 total_calls_make;
  optional int64 field_id=-1 unique_receivers;
  optional group field_id=-1 IDs_receivers_corr (List) {
    repeated group field_id=-1 list {
      optional binary field_id=-1 element (String);
    }
  }
  optional group field_id=-1 q_calls_corr (List) {
    repeated group field_id=-1 list {
      optional int64 field_id=-1 element;
    }
  }
  optional group field_id=-1 residence_distance_km_corr (List) {
    repeated group field_id=-1 list {
      optional double field_id=-1 element;
    }
  }
  optional group field_id=-1 calls_duration_total_corr (List) {
    repeated group field_id=-1 list {
      optional double field_id=-1 element;
    }
  }
}

Linhas: 2,968
Grupos de linhas: 1


In [ ]:
lavras = parquet_file.read().to_pandas()
lavras.head()

,city_residences,id_emisor,total_calls_make,unique_receivers,IDs_receivers_corr,q_calls_corr,residence_distance_km_corr,calls_duration_total_corr
0,Lavras,6A25BB13E0EB57AF0BC1B12E09066090,3,1,[09733364070D428DB7DB059EB964AEEA],[3],[1.6244811412788938],[8.469999999999999]
1,Lavras,A1F433373241547B7E7D0E4E63A8574F,27,1,[EAFB8A6F8D9AD7E91529F6EF7F15BF2B],[27],[1.6244811412788938],[19.590000000000003]
2,Lavras,C0EE955E574B587D037F1DC493ADDEC4,1,1,[54AAE028CCC31C4B800CB1E7E9799253],[1],[3.0086606658789745],[1.43]
3,Lavras,66406B8907461FBB978468BE0D0E22F1,3,1,[033183C2EE18E03EFDF1083D53EE92B5],[3],[1.5927049003955993],[29.29]
4,Lavras,EB4EF78A3B0BAAD118885F2B93881B5F,22,2,"[3F895921B209C4161B721382802BD3BC, 3DB84B3C732D58C6DE17A63C3C5618C0]","[21, 1]","[0.0, 0.0]","[18.19, 0.97]"


## Qualidade e consistencia

A validacao abaixo verifica nulos, duplicatas de emissor e consistencia entre a quantidade declarada de receptores e os comprimentos das listas. Essa etapa e importante porque as analises de rede dependem da correta correspondencia entre as listas.

In [ ]:
resumo_colunas = pd.DataFrame({
    'tipo': lavras.dtypes.astype(str),
    'nulos': lavras.isna().sum(),
    'n_unicos': [lavras[col].astype(str).nunique() for col in lavras.columns],
})
resumo_colunas

,tipo,nulos,n_unicos
city_residences,str,0,1
id_emisor,str,0,2968
total_calls_make,int64,0,234
unique_receivers,int64,0,36
IDs_receivers_corr,object,0,2911
q_calls_corr,object,0,1927
residence_distance_km_corr,object,0,1488
calls_duration_total_corr,object,0,2775


In [ ]:
list_cols = ['IDs_receivers_corr', 'q_calls_corr', 'residence_distance_km_corr', 'calls_duration_total_corr']

for col in list_cols:
    lavras[f'n_{col}'] = lavras[col].apply(len)

checks = pd.DataFrame({
    'linhas': [len(lavras)],
    'emissores_unicos': [lavras['id_emisor'].nunique()],
    'emissores_duplicados': [lavras['id_emisor'].duplicated().sum()],
    'cidades': [', '.join(sorted(lavras['city_residences'].unique()))],
    'unique_receivers_confere': [(lavras['unique_receivers'] == lavras['n_IDs_receivers_corr']).all()],
    'listas_mesmo_tamanho': [(lavras[[f'n_{c}' for c in list_cols]].nunique(axis=1) == 1).all()],
})
checks

,linhas,emissores_unicos,emissores_duplicados,cidades,unique_receivers_confere,listas_mesmo_tamanho
0,2968,2968,0,Lavras,True,True


## Estatisticas descritivas dos emissores

Aqui olhamos para o comportamento por emissor: volume total de chamadas, diversidade de contatos e concentracao. As medidas em log ajudam porque dados de chamadas geralmente possuem cauda longa.

In [ ]:
emissor_cols = ['total_calls_make', 'unique_receivers']
lavras[emissor_cols].describe(percentiles=[.01, .05, .25, .5, .75, .9, .95, .99]).T

,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,max
total_calls_make,2968.0,39.405323,53.080334,1.0,1.0,1.0,6.0,20.0,51.0,101.0,144.3,243.99,581.0
unique_receivers,2968.0,4.188679,4.032324,1.0,1.0,1.0,2.0,3.0,5.0,8.0,11.0,20.00,40.0


In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=lavras['total_calls_make'], nbinsx=60, name='Chamadas'))
fig.update_layout(title='Distribuicao do total de chamadas por emissor', xaxis_title='Total de chamadas', yaxis_title='Emissores')
fig.show()

fig = go.Figure()
fig.add_trace(go.Histogram(x=np.log1p(lavras['total_calls_make']), nbinsx=60, name='log1p chamadas'))
fig.update_layout(title='Distribuicao do total de chamadas por emissor em escala log1p', xaxis_title='log(1 + total de chamadas)', yaxis_title='Emissores')
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=lavras['unique_receivers'], nbinsx=40, name='Receptores unicos'))
fig.update_layout(title='Distribuicao de receptores distintos por emissor', xaxis_title='Receptores distintos', yaxis_title='Emissores')
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lavras['unique_receivers'],
    y=lavras['total_calls_make'],
    mode='markers',
    marker=dict(size=7, opacity=0.55),
    text=lavras['id_emisor'],
))
fig.update_layout(title='Total de chamadas vs. receptores distintos', xaxis_title='Receptores distintos', yaxis_title='Total de chamadas')
fig.show()

## Pareto de volume de chamadas

Esta analise mostra quanto do volume total de chamadas e explicado pelos emissores mais ativos. Se a curva acumular rapidamente, poucos usuarios concentram grande parte das chamadas.

In [ ]:
pareto = lavras[['id_emisor', 'total_calls_make', 'unique_receivers']].sort_values('total_calls_make', ascending=False).reset_index(drop=True)
pareto['rank'] = np.arange(1, len(pareto) + 1)
pareto['pct_emissores'] = pareto['rank'] / len(pareto)
pareto['pct_chamadas_acum'] = pareto['total_calls_make'].cumsum() / pareto['total_calls_make'].sum()
pareto.head(15)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=pareto['pct_emissores'], y=pareto['pct_chamadas_acum'], mode='lines'))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', line=dict(dash='dash'), name='Distribuicao uniforme'))
fig.update_layout(title='Curva de Pareto das chamadas', xaxis_title='Percentual acumulado de emissores', yaxis_title='Percentual acumulado de chamadas')
fig.show()

for corte in [0.01, 0.05, 0.10, 0.20]:
    n = max(1, int(np.ceil(corte * len(pareto))))
    pct = pareto.loc[:n-1, 'total_calls_make'].sum() / pareto['total_calls_make'].sum()
    print(f'Top {corte:.0%} dos emissores: {pct:.1%} das chamadas')

## Transformacao para arestas emissor-receptor

As listas do parquet sao expandidas para uma linha por par emissor-receptor. Essa estrutura facilita analises de distancia, duracao e rede.

In [ ]:
edges_records = []

for row in lavras.itertuples(index=False):
    for receiver, q_calls, distance_km, duration_total in zip(
        row.IDs_receivers_corr,
        row.q_calls_corr,
        row.residence_distance_km_corr,
        row.calls_duration_total_corr,
    ):
        edges_records.append({
            'city_residences': row.city_residences,
            'id_emisor': row.id_emisor,
            'id_receiver': receiver,
            'q_calls': q_calls,
            'residence_distance_km': distance_km,
            'calls_duration_total': duration_total,
        })

edges = pd.DataFrame(edges_records)
edges['avg_duration_per_call'] = edges['calls_duration_total'] / edges['q_calls'].replace(0, np.nan)
edges.head()

In [ ]:
pd.DataFrame({
    'arestas': [len(edges)],
    'emissores': [edges['id_emisor'].nunique()],
    'receptores': [edges['id_receiver'].nunique()],
    'pares_duplicados': [edges.duplicated(['id_emisor', 'id_receiver']).sum()],
    'chamadas_total_edges': [edges['q_calls'].sum()],
    'chamadas_total_emissores': [lavras['total_calls_make'].sum()],
})

## Distancia, duracao e intensidade das arestas

A distancia residencial entre emissor e receptor permite observar se a comunicacao e majoritariamente local ou se ha muitos vinculos de longa distancia. A duracao media por chamada ajuda a diferenciar contatos frequentes e curtos de contatos menos frequentes e longos.

In [ ]:
edges[['q_calls', 'residence_distance_km', 'calls_duration_total', 'avg_duration_per_call']].describe(percentiles=[.01, .05, .25, .5, .75, .9, .95, .99]).T

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=edges['residence_distance_km'], nbinsx=80))
fig.update_layout(title='Distribuicao da distancia residencial por aresta', xaxis_title='Distancia residencial (km)', yaxis_title='Arestas')
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=edges['residence_distance_km'],
    y=edges['q_calls'],
    mode='markers',
    marker=dict(size=6, opacity=0.35),
))
fig.update_layout(title='Intensidade de chamadas por distancia', xaxis_title='Distancia residencial (km)', yaxis_title='Quantidade de chamadas')
fig.show()

In [ ]:
bins = [-np.inf, 1, 5, 10, 25, 50, 100, 250, 500, np.inf]
labels = ['<=1', '1-5', '5-10', '10-25', '25-50', '50-100', '100-250', '250-500', '>500']
edges['faixa_distancia_km'] = pd.cut(edges['residence_distance_km'], bins=bins, labels=labels)

dist_summary = edges.groupby('faixa_distancia_km', observed=False).agg(
    arestas=('id_receiver', 'size'),
    chamadas=('q_calls', 'sum'),
    duracao_total=('calls_duration_total', 'sum'),
    chamadas_mediana=('q_calls', 'median'),
).reset_index()
dist_summary['pct_arestas'] = dist_summary['arestas'] / dist_summary['arestas'].sum()
dist_summary['pct_chamadas'] = dist_summary['chamadas'] / dist_summary['chamadas'].sum()
dist_summary

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=dist_summary['faixa_distancia_km'].astype(str), y=dist_summary['pct_arestas'], name='Arestas'))
fig.add_trace(go.Bar(x=dist_summary['faixa_distancia_km'].astype(str), y=dist_summary['pct_chamadas'], name='Chamadas'))
fig.update_layout(title='Participacao por faixa de distancia', xaxis_title='Faixa de distancia (km)', yaxis_title='Participacao', barmode='group')
fig.show()

## Concentracao de contatos por emissor

A proporcao de chamadas para o principal receptor indica dependencia de um contato dominante. Valores proximos de 1 sugerem que quase todas as chamadas do emissor foram para uma unica pessoa.

In [ ]:
concentracao = edges.groupby('id_emisor').agg(
    chamadas_total=('q_calls', 'sum'),
    receptores=('id_receiver', 'nunique'),
    maior_volume_receptor=('q_calls', 'max'),
    distancia_media_ponderada=('residence_distance_km', lambda s: np.average(s, weights=edges.loc[s.index, 'q_calls'])),
).reset_index()
concentracao['share_top_receptor'] = concentracao['maior_volume_receptor'] / concentracao['chamadas_total']
concentracao.sort_values('share_top_receptor', ascending=False).head(10)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=concentracao['share_top_receptor'], nbinsx=40))
fig.update_layout(title='Concentracao de chamadas no principal receptor', xaxis_title='Share do principal receptor', yaxis_title='Emissores')
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=concentracao['receptores'],
    y=concentracao['share_top_receptor'],
    mode='markers',
    marker=dict(size=7, opacity=0.55),
))
fig.update_layout(title='Diversidade de contatos vs. concentracao', xaxis_title='Receptores distintos', yaxis_title='Share do principal receptor')
fig.show()

## Analise de rede

A rede abaixo e direcionada: cada aresta sai de `id_emisor` para `id_receiver`, ponderada por `q_calls`. Como o parquet esta centrado nos emissores residentes em Lavras, muitos receptores podem aparecer apenas como destino, sem suas proprias chamadas de saida nesta base.

In [ ]:
G = nx.from_pandas_edgelist(
    edges,
    source='id_emisor',
    target='id_receiver',
    edge_attr=['q_calls', 'residence_distance_km', 'calls_duration_total'],
    create_using=nx.DiGraph,
)

network_summary = pd.DataFrame({
    'nos': [G.number_of_nodes()],
    'arestas': [G.number_of_edges()],
    'componentes_fracamente_conectados': [nx.number_weakly_connected_components(G)],
    'densidade': [nx.density(G)],
})
network_summary

In [ ]:
graus = pd.DataFrame({
    'id': list(G.nodes()),
    'in_degree': [G.in_degree(n) for n in G.nodes()],
    'out_degree': [G.out_degree(n) for n in G.nodes()],
    'weighted_in_calls': [G.in_degree(n, weight='q_calls') for n in G.nodes()],
    'weighted_out_calls': [G.out_degree(n, weight='q_calls') for n in G.nodes()],
})

graus.sort_values('weighted_in_calls', ascending=False).head(15)

In [ ]:
top_edges = edges.sort_values('q_calls', ascending=False).head(15)
top_edges[['id_emisor', 'id_receiver', 'q_calls', 'residence_distance_km', 'calls_duration_total', 'avg_duration_per_call']]

## Amostra de `residencias.csv`

A base de residencias e grande. A funcao abaixo le apenas as primeiras `n` linhas para uma visao rapida. Para estimativas mais estaveis, aumente `nrows` ou use leitura por chunks agregando contagens.

In [ ]:
residencias_sample = pd.read_csv(
    RESIDENCIAS_PATH,
    nrows=200_000,
    usecols=['ID', 'residence_city', 'residence_quintile_state', 'residence_quintile_nation'],
)

residencias_sample.head()

In [ ]:
cidade_top = residencias_sample['residence_city'].value_counts().head(15).reset_index()
cidade_top.columns = ['cidade', 'usuarios']
cidade_top

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=cidade_top['usuarios'], y=cidade_top['cidade'], orientation='h'))
fig.update_layout(title='Cidades mais frequentes na amostra de residencias', xaxis_title='Usuarios na amostra', yaxis_title='Cidade', yaxis=dict(autorange='reversed'))
fig.show()

quintis = pd.concat([
    residencias_sample['residence_quintile_state'].value_counts().sort_index().rename('quintil_estado'),
    residencias_sample['residence_quintile_nation'].value_counts().sort_index().rename('quintil_nacional'),
], axis=1).fillna(0).astype(int)
quintis

## Cruzamento futuro com quintis

Um proximo passo relevante e filtrar `residencias.csv` apenas para IDs presentes em `Lavras.parquet`, tanto emissores quanto receptores. Com isso, sera possivel comparar chamadas por quintil socioeconomico e investigar homofilia: pessoas tendem a se comunicar mais com usuarios do mesmo quintil?

In [ ]:
ids_interesse = set(lavras['id_emisor']) | set(edges['id_receiver'])
len(ids_interesse)

In [ ]:
def filtrar_residencias_por_ids(path, ids, chunksize=500_000):
    partes = []
    cols = ['ID', 'residence_city', 'residence_quintile_state', 'residence_quintile_nation']
    for chunk in pd.read_csv(path, usecols=cols, chunksize=chunksize):
        filtrado = chunk[chunk['ID'].isin(ids)]
        if not filtrado.empty:
            partes.append(filtrado.copy())
    if not partes:
        return pd.DataFrame(columns=cols)
    return pd.concat(partes, ignore_index=True).drop_duplicates('ID')

# Descomente para executar o cruzamento completo. Pode levar alguns minutos por causa do tamanho do CSV.
# residencias_ids = filtrar_residencias_por_ids(RESIDENCIAS_PATH, ids_interesse)
# residencias_ids.head()

## Leituras iniciais para relatorio

- O parquet de Lavras e pequeno o suficiente para analise em memoria e ja vem em formato de lista por emissor.
- A transformacao para arestas e a principal etapa para sair de estatisticas individuais e chegar em analises de rede.
- O total de chamadas provavelmente tem cauda longa; por isso, medias devem ser interpretadas junto com mediana e percentis.
- A distancia residencial permite separar interacoes locais e nao locais.
- A base de residencias abre caminho para analises socioeconomicas, mas precisa ser filtrada por IDs para evitar custo desnecessario.